# **RNN Model**

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding

# 1. Setup Data
train_sentences = [
    'The movie was truly amazing and exciting',   # Positive (1)
    'I absolutely loved the delicious food',      # Positive (1)
    'This is the best day ever',                 # Positive (1)
    'The service was terrible and very slow',    # Negative (0)
    'I really hated the boring movie',           # Negative (0)
    'This is the worst experience ever'          # Negative (0)
]
train_labels = np.array([1,1,1,0,0,0])

# 2. Pipeline
# A. Tokenize & Integer Encode
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(train_sentences)
sequences = tokenizer.texts_to_sequences(train_sentences)

# B. Padding
max_length = 8
padded_data = pad_sequences(sequences, maxlen=max_length, padding="post")

print("Word Index:", tokenizer.word_index)
print("Padded Data:\n", padded_data)

# 3. The model
model = Sequential()

# Layer 1: Embeding(The DNA Learner)
model.add(Embedding(input_dim=100, output_dim=8))

# Layer 2: SimpleRNN (The Sequence Reader)
model.add(SimpleRNN(16))

# Layer 3: Output (Decision)
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# 4. Train
model.fit(padded_data, train_labels, epochs=50)

# 5. Predict
test_text = ["I loved the exciting movie", "The food was terrible"]
test_seq = tokenizer.texts_to_sequences(test_text)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")

print("\n--- Test Results ---")
preds = model.predict(test_pad)
print("Predictions:", preds)

In [ ]:
import tensorflow as tf
import numpy as np
import random
import string

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

class TextGeneratorRNN:
    """RNN model for text generation using TensorFlow"""

    def __init__(self, text, seq_length=100, vocab_size=None, embedding_dim=256,
                 rnn_units=512, batch_size=64, buffer_size=10000):
        """
        Initialize the text generator.

        Args:
            text: Input text for training
            seq_length: Length of input sequences
            vocab_size: Size of vocabulary (auto-detected if None)
            embedding_dim: Dimension of embedding layer
            rnn_units: Number of RNN units
            batch_size: Batch size for training
            buffer_size: Buffer size for shuffling
        """
        self.text = text.lower()
        self.seq_length = seq_length
        self.embedding_dim = embedding_dim
        self.rnn_units = rnn_units
        self.batch_size = batch_size
        self.buffer_size = buffer_size

        # Create vocabulary
        self.vocab = sorted(set(self.text))
        self.vocab_size = len(self.vocab) if vocab_size is None else vocab_size

        # Create mapping dictionaries
        self.char2idx = {char: idx for idx, char in enumerate(self.vocab)}
        self.idx2char = np.array(self.vocab)

        # Create datasets
        self._prepare_datasets()

        # Build model
        self.model = self._build_model()

    def _prepare_datasets(self):
        """Convert text to numerical representation and create datasets"""
        # Convert text to integer indices
        self.text_as_int = np.array([self.char2idx[char] for char in self.text])

        # Create training examples
        self.char_dataset = tf.data.Dataset.from_tensor_slices(self.text_as_int)

        # Create sequences
        self.sequences = self.char_dataset.batch(self.seq_length + 1, drop_remainder=True)

        # Create input and target sequences
        def split_input_target(chunk):
            input_text = chunk[:-1]
            target_text = chunk[1:]
            return input_text, target_text

        self.dataset = self.sequences.map(split_input_target)

        # Shuffle, batch, and prefetch
        # Change to this:
        self.dataset = self.dataset.shuffle(self.buffer_size).batch(
            self.batch_size, drop_remainder=True
        ).prefetch(tf.data.AUTOTUNE)  # Remove .experimental

    def _build_model(self):
      """Build the RNN model"""
      model = tf.keras.Sequential([
          # Change this line - remove batch_input_shape
          tf.keras.layers.Embedding(self.vocab_size, self.embedding_dim,
                                    input_length=None),  # Remove batch_input_shape

          # Keep the rest the same
          tf.keras.layers.GRU(self.rnn_units,
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform',
                            dropout=0.2,
                            recurrent_dropout=0.2),

          tf.keras.layers.GRU(self.rnn_units,
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform',
                            dropout=0.2,
                            recurrent_dropout=0.2),

          tf.keras.layers.Dense(self.vocab_size)
      ])

      return model

    def train(self, epochs=50, learning_rate=0.001):
        """Train the model"""
        # Compile model
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        )

        # Callbacks
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
          filepath='rnn_text_generator_checkpoint.weights.h5',  # Add .weights.h5 extension
          save_weights_only=True,
          save_best_only=True,
          monitor='loss',
          # mode='min',
          verbose=1
      )

        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        )

        # Train
        history = self.model.fit(
            self.dataset,
            epochs=epochs,
            callbacks=[checkpoint_callback, early_stopping],
            verbose=1
        )

        return history

    def generate_text(self, start_string, num_generate=500, temperature=1.0):
        """
        Generate text using the trained model.

        Args:
            start_string: String to start generation
            num_generate: Number of characters to generate
            temperature: Controls randomness (lower = more deterministic)

        Returns:
            Generated text
        """
        # Convert start string to indices
        input_eval = [self.char2idx[s] for s in start_string.lower()]
        input_eval = tf.expand_dims(input_eval, 0)

        text_generated = []

        # Store original batch size
        original_batch_size = self.batch_size

        # Temporarily change batch size for generation
        self.model = self._build_model_with_batch_size(1)

        # Load weights if available
        # Load weights if available
        try:
            self.model.load_weights('rnn_text_generator_checkpoint.weights.h5')  # Add .weights.h5 extension
        except:
            pass

        # Generate characters
        for _ in range(num_generate):
            # Get predictions
            predictions = self.model(input_eval)

            # Remove batch dimension
            predictions = predictions[0, -1, :]

            # Apply temperature
            predictions = predictions / temperature
            predictions = tf.nn.softmax(predictions).numpy()

            # Sample from the distribution
            predicted_id = np.random.choice(len(predictions), p=predictions)

            # Append to generated text
            text_generated.append(self.idx2char[predicted_id])

            # Update input for next iteration
            input_eval = tf.expand_dims([predicted_id], 0)

        # Restore original batch size
        self.model = self._build_model_with_batch_size(original_batch_size)

        return start_string + ''.join(text_generated)

    def _build_model_with_batch_size(self, batch_size):
      """Build model with a specific batch size for generation"""
      model = tf.keras.Sequential([
          # Change this line
          tf.keras.layers.Embedding(self.vocab_size, self.embedding_dim,
                                    input_length=None),  # Remove batch_input_shape
          tf.keras.layers.GRU(self.rnn_units,
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform'),
          tf.keras.layers.GRU(self.rnn_units,
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform'),
          tf.keras.layers.Dense(self.vocab_size)
      ])
      return model

    def save_model(self, filepath='rnn_text_generator_model'):
        """Save the trained model"""
        self.model.save(filepath)

    def load_model(self, filepath='rnn_text_generator_model'):
        """Load a saved model"""
        self.model = tf.keras.models.load_model(filepath)


# Example usage with sample text
def main():
    # Sample text for demonstration (you can replace with any text file)
    sample_text = """
    The quick brown fox jumps over the lazy dog. This is a simple example of text generation.
    Machine learning models can learn patterns from text data and generate new content.
    The generated text might not always make perfect sense, but it can be surprisingly creative.
    As the model trains longer, it learns better grammar, word relationships, and style.
    RNNs are particularly good at sequence prediction tasks like text generation.
    """

    # Initialize the text generator
    print("Initializing text generator...")
    generator = TextGeneratorRNN(
        text=sample_text,
        seq_length=50,        # Length of input sequences
        embedding_dim=128,    # Embedding dimension
        rnn_units=256,        # Number of RNN units
        batch_size=8,        # Batch size
    )

    # Train the model
    print("\nTraining model...")
    history = generator.train(epochs=30, learning_rate=0.001)

    # Generate text
    print("\nGenerating text...")
    generated_text = generator.generate_text(
        start_string="The machine",
        num_generate=200,
        temperature=0.8  # Adjust for more/less randomness
    )
    print("\nGenerated Text:")
    print(generated_text)

    # Generate with different temperatures for comparison
    print("\n\nGenerating with different temperatures:")
    for temp in [0.5, 1.0, 1.5]:
        print(f"\n--- Temperature = {temp} ---")
        text = generator.generate_text(
            start_string="The model",
            num_generate=100,
            temperature=temp
        )
        print(text)


# Simplified version for quick testing
class SimpleRNNTextGenerator:
    """Simpler RNN text generator for quick prototyping"""

    def __init__(self, text, sequence_length=100):
        self.text = text.lower()
        self.seq_length = sequence_length

        # Create character mappings
        self.chars = sorted(list(set(text)))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)

        # Build model
        self.model = self._build_simple_model()

    def _build_simple_model(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Embedding(self.vocab_size, 64, input_length=self.seq_length),
            tf.keras.layers.SimpleRNN(128, return_sequences=True, dropout=0.2),
            tf.keras.layers.SimpleRNN(128, dropout=0.2),
            tf.keras.layers.Dense(self.vocab_size, activation='softmax')
        ])
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy'
        )
        return model

    def prepare_data(self):
        """Prepare input sequences and targets"""
        input_seqs = []
        targets = []

        for i in range(0, len(self.text) - self.seq_length, 1):
            input_seq = self.text[i:i + self.seq_length]
            target_char = self.text[i + self.seq_length]

            input_seqs.append([self.char_to_idx[ch] for ch in input_seq])
            targets.append(self.char_to_idx[target_char])

        return np.array(input_seqs), np.array(targets)

    def train(self, epochs=20, batch_size=128):
        X, y = self.prepare_data()
        self.model.fit(X, y, batch_size=batch_size, epochs=epochs, validation_split=0.1)

    def generate(self, seed_text, length=200):
        generated = seed_text.lower()

        for _ in range(length):
            # Take last seq_length characters
            input_text = generated[-self.seq_length:]

            # Convert to indices
            input_seq = np.array([[self.char_to_idx.get(ch, 0) for ch in input_text]])

            # Predict
            predicted = self.model.predict(input_seq, verbose=0)[0]
            predicted_idx = np.argmax(predicted)

            # Append predicted character
            generated += self.idx_to_char[predicted_idx]

        return generated


if __name__ == "__main__":
    # Run the main example
    main()

    # Quick test with simple version
    print("\n\n" + "="*50)
    print("QUICK TEST WITH SIMPLE RNN")
    print("="*50)

    sample = "hello world this is a test of text generation using recurrent neural networks"
    simple_gen = SimpleRNNTextGenerator(sample, sequence_length=20)
    simple_gen.train(epochs=10)
    print("\nGenerated text:")
    print(simple_gen.generate("hello", length=100))

In [ ]:
import tensorflow as tf
import numpy as np
import random
import string

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

class TextGeneratorLSTM:
    """LSTM model for text generation using TensorFlow"""

    def __init__(self, text, seq_length=100, vocab_size=None, embedding_dim=256,
                 lstm_units=512, batch_size=64, buffer_size=10000):
        """
        Initialize the text generator.

        Args:
            text: Input text for training
            seq_length: Length of input sequences
            vocab_size: Size of vocabulary (auto-detected if None)
            embedding_dim: Dimension of embedding layer
            lstm_units: Number of lstm units
            batch_size: Batch size for training
            buffer_size: Buffer size for shuffling
        """
        self.text = text.lower()
        self.seq_length = seq_length
        self.embedding_dim = embedding_dim
        self.lstm_units = lstm_units
        self.batch_size = batch_size
        self.buffer_size = buffer_size

        # Create vocabulary
        self.vocab = sorted(set(self.text))
        self.vocab_size = len(self.vocab) if vocab_size is None else vocab_size

        # Create mapping dictionaries
        self.char2idx = {char: idx for idx, char in enumerate(self.vocab)}
        self.idx2char = np.array(self.vocab)

        # Create datasets
        self._prepare_datasets()

        # Build model
        self.model = self._build_model()

    def _prepare_datasets(self):
        """Convert text to numerical representation and create datasets"""
        # Convert text to integer indices
        self.text_as_int = np.array([self.char2idx[char] for char in self.text])

        # Create training examples
        self.char_dataset = tf.data.Dataset.from_tensor_slices(self.text_as_int)

        # Create sequences
        self.sequences = self.char_dataset.batch(self.seq_length + 1, drop_remainder=True)

        # Create input and target sequences
        def split_input_target(chunk):
            input_text = chunk[:-1]
            target_text = chunk[1:]
            return input_text, target_text

        self.dataset = self.sequences.map(split_input_target)

        # Shuffle, batch, and prefetch
        # Change to this:
        self.dataset = self.dataset.shuffle(self.buffer_size).batch(
            self.batch_size, drop_remainder=True
        ).prefetch(tf.data.AUTOTUNE)  # Remove .experimental

    def _build_model(self):
      """Build the LSTM model"""
      model = tf.keras.Sequential([
          tf.keras.layers.Embedding(self.vocab_size, self.embedding_dim,
                                    input_length=None),

          # Change GRU to LSTM
          tf.keras.layers.LSTM(self.lstm_units,
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform',
                            dropout=0.2,
                            recurrent_dropout=0.2),

          # Second LSTM layer
          tf.keras.layers.LSTM(self.lstm_units,
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform',
                            dropout=0.2,
                            recurrent_dropout=0.2),

          tf.keras.layers.Dense(self.vocab_size)
      ])

      return model

    def train(self, epochs=50, learning_rate=0.001):
        """Train the model"""
        # Compile model
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        )

        # Callbacks
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
          filepath='lstm_text_generator_checkpoint.weights.h5',  # Add .weights.h5 extension
          save_weights_only=True,
          save_best_only=True,
          monitor='loss',
          # mode='min',
          verbose=1
      )

        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        )

        # Train
        history = self.model.fit(
            self.dataset,
            epochs=epochs,
            callbacks=[checkpoint_callback, early_stopping],
            verbose=1
        )

        return history

    def generate_text(self, start_string, num_generate=500, temperature=1.0):
        """
        Generate text using the trained model.

        Args:
            start_string: String to start generation
            num_generate: Number of characters to generate
            temperature: Controls randomness (lower = more deterministic)

        Returns:
            Generated text
        """
        # Convert start string to indices
        input_eval = [self.char2idx[s] for s in start_string.lower()]
        input_eval = tf.expand_dims(input_eval, 0)

        text_generated = []

        # Store original batch size
        original_batch_size = self.batch_size

        # Temporarily change batch size for generation
        self.model = self._build_model_with_batch_size(1)

        # Load weights if available
        # Load weights if available
        try:
            self.model.load_weights('lstm_text_generator_checkpoint.weights.h5')  # Add .weights.h5 extension
        except:
            pass

        # Generate characters
        for _ in range(num_generate):
            # Get predictions
            predictions = self.model(input_eval)

            # Remove batch dimension
            predictions = predictions[0, -1, :]

            # Apply temperature
            predictions = predictions / temperature
            predictions = tf.nn.softmax(predictions).numpy()

            # Sample from the distribution
            predicted_id = np.random.choice(len(predictions), p=predictions)

            # Append to generated text
            text_generated.append(self.idx2char[predicted_id])

            # Update input for next iteration
            input_eval = tf.expand_dims([predicted_id], 0)

        # Restore original batch size
        self.model = self._build_model_with_batch_size(original_batch_size)

        return start_string + ''.join(text_generated)

    def _build_model_with_batch_size(self, batch_size):
      """Build model with a specific batch size for generation"""
      model = tf.keras.Sequential([
          tf.keras.layers.Embedding(self.vocab_size, self.embedding_dim,
                                    input_length=None),
          # Change GRU to LSTM
          tf.keras.layers.LSTM(self.lstm_units,
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform'),
          tf.keras.layers.LSTM(self.lstm_units,
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform'),
          tf.keras.layers.Dense(self.vocab_size)
      ])
      return model

    def save_model(self, filepath='lstm_text_generator_model'):
        """Save the trained model"""
        self.model.save(filepath)

    def load_model(self, filepath='lstm_text_generator_model'):
        """Load a saved model"""
        self.model = tf.keras.models.load_model(filepath)


# Example usage with sample text
def main():
    # Sample text for demonstration (you can replace with any text file)
    sample_text = """
    The quick brown fox jumps over the lazy dog. This is a simple example of text generation.
    Machine learning models can learn patterns from text data and generate new content.
    The generated text might not always make perfect sense, but it can be surprisingly creative.
    As the model trains longer, it learns better grammar, word relationships, and style.
    LSTMs are particularly good at sequence prediction tasks like text generation.
    """

    # Initialize the text generator
    print("Initializing text generator...")
    generator = TextGeneratorLSTM(
        text=sample_text,
        seq_length=50,        # Length of input sequences
        embedding_dim=128,    # Embedding dimension
        lstm_units=256,        # Number of LSTM units
        batch_size=8,        # Batch size
    )

    # Train the model
    print("\nTraining model...")
    history = generator.train(epochs=50, learning_rate=0.001)

    # Generate text
    print("\nGenerating text...")
    generated_text = generator.generate_text(
        start_string="The machine",
        num_generate=200,
        temperature=0.8  # Adjust for more/less randomness
    )
    print("\nGenerated Text:")
    print(generated_text)

    # Generate with different temperatures for comparison
    print("\n\nGenerating with different temperatures:")
    for temp in [0.5, 1.0, 1.5]:
        print(f"\n--- Temperature = {temp} ---")
        text = generator.generate_text(
            start_string="The model",
            num_generate=100,
            temperature=temp
        )
        print(text)


# Simplified version for quick testing
class SimpleLSTMTextGenerator:
    """Simpler LSTM text generator for quick prototyping"""

    def __init__(self, text, sequence_length=100):
        self.text = text.lower()
        self.seq_length = sequence_length

        # Create character mappings
        self.chars = sorted(list(set(text)))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)

        # Build model
        self.model = self._build_simple_model()

    def _build_simple_model(self):
      model = tf.keras.Sequential([
          tf.keras.layers.Embedding(self.vocab_size, 128, input_length=self.seq_length),  # 64->128
          tf.keras.layers.LSTM(256, return_sequences=True, dropout=0.2),  # 128->256
          tf.keras.layers.LSTM(256, dropout=0.2),  # 128->256
          tf.keras.layers.Dense(self.vocab_size, activation='softmax')
      ])
      model.compile(
          optimizer='adam',
          loss='sparse_categorical_crossentropy'
      )
      return model

    def prepare_data(self):
        """Prepare input sequences and targets"""
        input_seqs = []
        targets = []

        for i in range(0, len(self.text) - self.seq_length, 1):
            input_seq = self.text[i:i + self.seq_length]
            target_char = self.text[i + self.seq_length]

            input_seqs.append([self.char_to_idx[ch] for ch in input_seq])
            targets.append(self.char_to_idx[target_char])

        return np.array(input_seqs), np.array(targets)

    def train(self, epochs=20, batch_size=128):
        X, y = self.prepare_data()
        self.model.fit(X, y, batch_size=batch_size, epochs=epochs, validation_split=0.1)

    def generate(self, seed_text, length=200):
        generated = seed_text.lower()

        for _ in range(length):
            # Take last seq_length characters
            input_text = generated[-self.seq_length:]

            # Convert to indices
            input_seq = np.array([[self.char_to_idx.get(ch, 0) for ch in input_text]])

            # Predict
            predicted = self.model.predict(input_seq, verbose=0)[0]
            predicted_idx = np.argmax(predicted)

            # Append predicted character
            generated += self.idx_to_char[predicted_idx]

        return generated


if __name__ == "__main__":
    # Run the main example
    main()

    # Quick test with simple version
    print("\n\n" + "="*50)
    print("QUICK TEST WITH SIMPLE LSTM")
    print("="*50)

    sample = "hello world this is a test of text generation using long short term memory " * 50
    simple_gen = SimpleLSTMTextGenerator(sample, sequence_length=20)
    simple_gen.train(epochs=10)
    print("\nGenerated text:")
    print(simple_gen.generate("hello", length=100))